# Clase 12 — Pandas. Clasificador de Morosidad con KNN. 

Hasta ahora usamos pandas para explorar y preparar datos. Hoy vamos a usar un dataset real de solicitudes de crédito para entrenar un modelo que responda una pregunta concreta: **¿esta solicitud corresponde a una persona morosa?**

La columna objetivo es `moroso`: `0` significa que no es moroso y `1` que sí lo es. El objetivo didáctico no es construir un sistema de crédito para producción, sino recorrer el flujo completo de clasificación con datos reales.

**Objetivos de la clase:**
- Cargar y explorar un CSV de créditos con pandas.
- Visualizar dos variables antes de entrenar.
- Preparar variables numéricas, incluyendo un valor faltante.
- Separar train/test y entrenar un clasificador KNN.
- Evaluar el modelo con una matriz de confusión y predecir una nueva solicitud.

---
## 1. Cargar y conocer el dataset

Cada fila representa una solicitud. Además de la edad y los ingresos, hay datos de deuda, cuotas, consultas y situación crediticia. Cargamos el archivo y miramos algunas filas antes de tomar decisiones.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

creditos = pd.read_csv("dataset_credito.csv")

print("Forma del dataset (filas, columnas):", creditos.shape)

display(creditos.head())

---
## 2. Exploración inicial de datos (EDA)

Antes de elegir un modelo conviene hacer una exploración breve y ordenada. El EDA (*Exploratory Data Analysis*) no busca responder todo: busca detectar problemas que podrían volver inválido el análisis o el entrenamiento.

Vamos a responder, en este orden:

1. ¿Qué columnas tenemos y qué tipo de dato interpreta pandas?
2. ¿Hay faltantes o filas duplicadas?
3. ¿Cómo se distribuyen los números y nuestra variable objetivo?
4. ¿Qué variables conviene usar en este primer modelo?

In [ ]:
# 2.1 Estructura y tipos de datos
# info() muestra cantidad de valores no nulos y el tipo inferido para cada columna.
creditos.info()

print("\nCantidad de columnas por tipo:")
display(creditos.dtypes.value_counts())

In [ ]:
# Las columnas object son texto/categorías: no pueden ir directamente a KNN.
columnas_texto = creditos.select_dtypes(include="object").columns.tolist()

print("Columnas de texto o categóricas:", columnas_texto)

### 2.2 Valores faltantes y duplicados

Un valor faltante no siempre implica borrar una fila. Primero medimos cuántos hay y qué proporción representan. Después podremos decidir cómo tratarlos; aquí el modelo usará la mediana para completar los faltantes de las variables seleccionadas. También revisamos duplicados exactos para evitar contar dos veces la misma observación.

In [ ]:
print(f"Filas duplicadas exactas: {creditos.duplicated().sum()}")

In [ ]:
faltantes = pd.DataFrame(
    {
    "cantidad": creditos.isna().sum(),
    "porcentaje": (creditos.isna().mean() * 100).round(2),
    }
)

faltantes = faltantes[faltantes["cantidad"] > 0].sort_values("cantidad", ascending=False)

print("Columnas con valores faltantes:")
display(faltantes)

### 2.3 Resumen de las variables numéricas

`describe()` permite detectar escalas muy distintas, valores extremos y rangos poco razonables. Por ejemplo, aquí veremos que los montos de deuda están en una escala mucho mayor que la edad: esa es la razón por la que más adelante estandarizaremos antes de aplicar KNN.

In [ ]:
creditos.describe().T

In [ ]:
creditos.hist(bins=10, figsize=(15, 15))

In [ ]:
# Creamos una lista con las variables de interés para el análisis exploratorio y la visualización.
variables_revision = [
    "edad", "ingresos", "Compromisos_Mensual", "Endeudamiento_Externo",
    "canti_moras", "dias_atraso", "Cuota", "Monto_Otorgado",
]

# Filtramos las variables de interés para el análisis exploratorio y la visualización.
creditos[variables_revision].describe()

In [ ]:
# Verificamos la cantidad de categorías únicas en cada columna de texto/categoría.
etiquetas_unicas = creditos[columnas_texto].nunique()

etiquetas_unicas = etiquetas_unicas.sort_values(ascending=False)

print("Cantidad de categorías en las columnas de texto:")
etiquetas_unicas

### 2.4 Distribución de la variable objetivo

Por último, verificamos cuántos casos hay de cada clase. Si una clase es mucho más frecuente, la *accuracy* puede parecer buena aunque el modelo ignore la clase menos común. Por eso, además de accuracy, después leeremos la matriz de confusión.

In [ ]:
distribucion_objetivo = creditos["moroso"].value_counts()

resumen_objetivo = pd.DataFrame({
    "cantidad": distribucion_objetivo,
    "porcentaje": (distribucion_objetivo / len(creditos) * 100).round(1),
})

display(resumen_objetivo)

print("Conclusión del EDA:")
print("• Hay variables numéricas y categóricas; en este primer KNN usaremos solo las numéricas.")
print("• Los faltantes numéricos se imputarán con la mediana dentro del Pipeline.")
print("• Las clases no están balanceadas, así que evaluaremos también los errores por clase.")

---
## 3. Visualizar antes de entrenar

Un buen hábito: **mirar los datos antes de entrenar cualquier modelo**. En este gráfico cada punto es una solicitud. La escala logarítmica en deuda permite ver mejor los casos con montos muy diferentes.

In [ ]:
# Definimos colores y etiquetas para la visualización de morosos y no morosos.
colores = {0: "steelblue", 1: "tomato"}
etiquetas = {0: "sin mora", 1: "moroso"}

# Generamos un gráfico de dispersión de ingresos vs. endeudamiento externo.
fig, ax = plt.subplots(figsize=(10, 5))
for clase, grupo in creditos.groupby("moroso"):
    ax.scatter(
        grupo["ingresos"], grupo["Endeudamiento_Externo"],
        label=etiquetas[clase], color=colores[clase]
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Ingresos")
ax.set_ylabel("Endeudamiento externo")
ax.set_title("Solicitudes de crédito: morosos y no morosos")
ax.legend()
plt.show()

---
## 4. Preparar los datos y entrenar un KNN

KNN decide según las distancias entre casos. Por eso las variables deben estar en una escala comparable: los ingresos están en miles, mientras que la edad está en decenas. 

Usamos un `Pipeline` que primero completa valores faltantes con la mediana, luego estandariza y finalmente entrena el clasificador.

Dejamos fuera `ID`, porque identifica una fila y no describe a la persona. También dejamos las columnas de texto para una clase posterior, donde veremos cómo codificarlas.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Definimos las variables predictoras.
variables = [
    "ingresos", "edad", "BCRA_Peor_Situacion",
    "Cantidad_consultas_7_dias", "Compromisos_Mensual",
    "Endeudamiento_Externo", "canti_moras", "dias_atraso",
    "Cuota", "cant_cuotas", "Monto_Otorgado",
]

X = creditos[variables] # Variables predictoras
y = creditos["moroso"]  # Variable objetivo

# Dividimos el dataset en train y test, estratificando por la variable objetivo.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Creamos un Pipeline que imputa los valores faltantes con la mediana, estandariza las variables y aplica KNN.
modelo = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7)),
])

# Entrenamos el modelo con los datos de entrenamiento
modelo.fit(X_train, y_train)

# Hacemos predicciones sobre el conjunto de prueba.
y_pred = modelo.predict(X_test)

print(f"Train: {X_train.shape[0]} solicitudes — Test: {X_test.shape[0]} solicitudes")
print("Primeras predicciones:", y_pred[:10])

---
## 5. La matriz de confusión, como gráfico

La accuracy sola puede engañarnos cuando hay muchas más solicitudes no morosas que morosas. La matriz de confusión muestra qué tipo de errores está cometiendo el modelo. En un caso real, decidir cuál error es más costoso requiere criterios del negocio y revisión humana.

In [ ]:
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix

exactitud = accuracy_score(y_test, y_pred)
matriz = confusion_matrix(y_test, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    matriz, annot=True, fmt="d", cmap="Oranges",
    xticklabels=["no moroso", "moroso"],
    yticklabels=["no moroso", "moroso"], ax=ax,
)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de confusión (accuracy = {exactitud:.2f})")
plt.show()

---
## 📝 Actividad — Clasificá una nueva solicitud

**Consigna:**
- Elegí valores plausibles para una nueva solicitud.
- Pedile al modelo que prediga si será `moroso` o `no moroso`.
- Probá cambiar solo `dias_atraso`, `canti_moras` o `Endeudamiento_Externo` y observá si cambia el resultado.


In [ ]:
# ACTIVIDAD: cambiar los valores de esta nueva solicitud
nueva_solicitud = pd.DataFrame([{
    "ingresos": 50000,
    "edad": 35,
    "BCRA_Peor_Situacion": 1,                   # Va desde 1 (sin mora) hasta 5 (incobrable)
    "Cantidad_consultas_7_dias": 2,
    "Compromisos_Mensual": 9000,
    "Endeudamiento_Externo": 80000,
    "canti_moras": 0,                          # Cantidad de moras en el historial crediticio
    "dias_atraso": 0,
    "Cuota": 5000,
    "cant_cuotas": 12,
    "Monto_Otorgado": 30000,
}])

prediccion = modelo.predict(nueva_solicitud)[0]
probabilidad = modelo.predict_proba(nueva_solicitud)[0, 1]

print("Predicción:", etiquetas[prediccion])
print(f"Proporción de vecinos morosos: {probabilidad:.0%}")